# Reproducible wrangling: script it as a pipeline

_Data Wrangling_

**Bundle every cleaning step into one fitted object so train, test, and live serving all get the exact same treatment.**

Cleaning data by clicking through notebook cells &mdash; fill this null, drop that row, map these
       categories &mdash; feels productive, but it leaves you with no record of what you did that you
       can replay. Next month's data arrives and you cannot reproduce the exact same cleaning. Worse, when
       the model goes live, the serving code does cleaning its own way, and the live data is now
       subtly different from the training data. The model degrades and nobody knows why.

       The fix is to stop hand-editing and instead script the cleaning as a pipeline: an ordered
       list of named steps that you fit once and can re-run on any data forever. In
       scikit-learn this is a Pipeline, and a ColumnTransformer lets you apply
       different steps to different column types &mdash; impute-then-scale the numeric columns,
       impute-then-one-hot-encode the categorical columns &mdash; all bundled into one object.

---

This notebook is a practice scaffold for the **Reproducible wrangling: script it as a pipeline** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + scikit-learn

In [ ]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# --- A small mixed-type table with missing values (NaN) ---
df = pd.DataFrame({
    "age":     [30, 50, None, 24, 41, 38, None, 55],   # numeric, has a gap
    "income":  [60, 85, 72, None, 90, 64, 51, 120],    # numeric, has a gap
    "city":    ["NYC", "LA", "NYC", "SF", None, "LA", "SF", "NYC"],  # categorical
    "plan":    ["free", "pro", "pro", "free", "pro", None, "free", "pro"],
    "churn":   [0, 1, 0, 0, 1, 0, 1, 1],
})
y = df.pop("churn")
X = df

numeric_cols     = ["age", "income"]
categorical_cols = ["city", "plan"]

# --- Per-column-type cleaning, bundled in a ColumnTransformer ---
numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),  # unseen live cats -> all-zeros
])
prep = ColumnTransformer([
    ("num", numeric_branch,     numeric_cols),
    ("cat", categorical_branch, categorical_cols),
])

# --- One Pipeline: cleaning THEN model ---
pipe = Pipeline([
    ("prep",  prep),
    ("model", RandomForestClassifier(n_estimators=200, random_state=0)),
])

# --- THE GOLDEN RULE: fit on TRAIN only; transform/predict on test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
pipe.fit(X_train, y_train)            # learns fills, mean/std, categories FROM TRAIN
print("test accuracy:", pipe.score(X_test, y_test))
preds = pipe.predict(X_test)          # cleans test with train-learned values, then predicts

# --- Persist ONE artifact: transforms + model together ---
joblib.dump(pipe, "churn_pipeline.joblib")

# --- Serving: load the same artifact, predict on a RAW live row ---
served = joblib.load("churn_pipeline.joblib")
live_row = pd.DataFrame([{"age": 47, "income": None, "city": "BERLIN", "plan": "pro"}])
# income is filled with the TRAIN median; 'BERLIN' is an unseen city -> handled, not a crash.
print("live prediction:", served.predict(live_row)[0])

## Visualize the data & results

_When you fit a feature selector + scaler on ALL the data (leaked) vs inside a Pipeline that refits on TRAIN only per fold, how do the reported accuracies compare? Shown on data where the TRUE accuracy is exactly chance (0.5), so any lift above 0.5 is pure leakage._

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

rng = np.random.RandomState(0)
n, p = 200, 5000
X = rng.randn(n, p)          # pure noise features
y = rng.randint(0, 2, n)     # RANDOM labels -> nothing learnable; true acc = 0.5

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# --- LEAKED: select the 20 "best" features using ALL the data (peeks at every label) ---
X_leak = SelectKBest(f_classif, k=20).fit_transform(StandardScaler().fit_transform(X), y)
acc_leak = cross_val_score(LogisticRegression(max_iter=2000), X_leak, y, cv=cv).mean()

# --- HONEST: identical steps inside a Pipeline -> refit per fold on TRAIN only ---
pipe = Pipeline([
    ("scale",  StandardScaler()),
    ("select", SelectKBest(f_classif, k=20)),
    ("model",  LogisticRegression(max_iter=2000)),
])
acc_honest = cross_val_score(pipe, X, y, cv=cv).mean()

print("leaked  CV accuracy:", round(acc_leak, 4))    # -> 0.81   (mirage)
print("honest  CV accuracy:", round(acc_honest, 4))  # -> 0.425  (~chance)
print("true accuracy       :", 0.5)                  # nothing is learnable

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A colleague standardizes all the features with StandardScaler().fit_transform(X) on the whole dataset, then splits into train and test and reports a great test accuracy. What is wrong, and how should it be done?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify that the scaler LEARNS the mean and standard deviation from data. — _Those are parameters of the transform, so fitting them is "training" and must respect the train/test boundary._
- Note that fitting on all of X uses the test rows to set the mean and std. — _Test information has bled into the transform applied to the training data &mdash; that is leakage, and it inflates the test score._
- Put the scaler inside a Pipeline with the model and call pipe.fit(X_train), then pipe.score(X_test). — _Now the mean/std are learned on train only and merely applied to test &mdash; honest by construction._

**Answer:** The StandardScaler was fit on the full dataset, so its mean and standard deviation depend on the test rows &mdash; classic leakage that inflates the reported test accuracy. The fix is to wrap the scaler and model in a Pipeline and call fit on the training set only; the pipeline then transforms the test set with the train-learned statistics. The good score was a mirage that would collapse on live data.

</details>

**Problem 2.** You have numeric columns with missing values and categorical columns with missing values. Sketch the scikit-learn object that cleans both correctly in one shot.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build a numeric branch: SimpleImputer(strategy="median") then StandardScaler(), chained in a small Pipeline. — _Numbers want a median fill (robust to outliers) and standardization._
- Build a categorical branch: SimpleImputer(strategy="most_frequent") then OneHotEncoder(handle_unknown="ignore"). — _Categories want a most-frequent fill and one-hot encoding; ignoring unknowns prevents crashes on unseen live categories._
- Combine the two branches with a ColumnTransformer, routing numeric column names to the first branch and categorical names to the second. — _The ColumnTransformer applies the right steps to the right columns and concatenates the results._

**Answer:** A ColumnTransformer with two branches: numeric columns go through SimpleImputer(strategy="median") &rarr; StandardScaler(), and categorical columns go through SimpleImputer(strategy="most_frequent") &rarr; OneHotEncoder(handle_unknown="ignore"). Drop that ColumnTransformer into a Pipeline as the "prep" step ahead of the model, and the whole cleaning-plus-model fits and transforms as one object.

</details>

**Problem 3.** Your training script saves the model with joblib.dump(model, ...) but re-implements the cleaning fresh inside the serving service. Why is this risky, and what should you save instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that the cleaning includes data-learned values: imputer fills, scaler mean/std, encoder category lists. — _Re-implementing the cleaning recomputes these from whatever data the serving service has &mdash; not the training values._
- See that this creates train/serve skew: the live row is cleaned with different numbers than training used. — _Even a small mismatch (a different fill, an unseen category handled differently) shifts the input distribution the model sees._
- Save the whole fitted pipeline &mdash; transforms and model together &mdash; with joblib.dump(pipe, "model.joblib"), and have serving joblib.load it and call .predict on raw rows. — _One artifact means training and serving share the exact same fitted cleaning._

**Answer:** Re-implementing the cleaning at serving time recomputes every data-learned value (fills, mean/std, category maps) from the wrong data, causing train/serve skew. Instead, save the entire fitted Pipeline (preprocessing + model) as one joblib artifact, and have the serving service load it and call .predict on raw rows. Training and production then run byte-for-byte identical cleaning.

</details>